 # Section 1 Import และ Path Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install mne -q

In [ ]:
import json
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

PROJECT_DIR = "/content/drive/MyDrive/SleepStageClassificationProject"
DATA_DIR = Path(f"{PROJECT_DIR}/dataset/HMC")
OUT_DIR = Path(f"{PROJECT_DIR}/dataset/preprocess_microvolt")

for name in ["train", "val", "test"]:
    (OUT_DIR / name).mkdir(parents=True, exist_ok=True)

SEED = 42
FS = 256
EPOCH_LEN = 30 * FS
N_CONSEC = 5
STAGES = ["W", "N1", "N2", "N3", "R"]

CHANNELS = [
    "F4-M1", "C4-M1", "O2-M1", "C3-M2",
    "E1-M2", "E2-M2", "chin"
]
CHANNEL_NAMES = [
    "F4-M1", "C4-M1", "O2-M1", "C3-M2",
    "E1-M2", "E2-M2", "Chin EMG"
]

EXCLUDED = ["SN014", "SN064", "SN135"]

# Section 2 Train / Validation / Test Subject Split

In [ ]:
split_path = OUT_DIR / "split_subjects.json"

if split_path.exists():
    with open(split_path) as f:
        split_info = json.load(f)

    current_subjects = sorted(
        p.stem for p in DATA_DIR.glob("SN*.edf")
        if "sleepscoring" not in p.stem and p.stem not in EXCLUDED
    )

    assert split_info["n_train"] + split_info["n_val"] + split_info["n_test"] == len(current_subjects), \
        "จำนวน Subject ใน split เดิมไม่ตรงกับข้อมูลปัจจุบัน"

    assert set(split_info["all_subjects"]) == set(current_subjects), \
        "Subject ใน split เดิมไม่ตรงกับข้อมูลปัจจุบัน"

    assert not set(EXCLUDED) & set(split_info["all_subjects"]), \
        "split เดิมมี Subject ที่ต้อง exclude"

    print("โหลด split เดิม")
else:
    all_subjects = sorted(p.stem for p in DATA_DIR.glob("SN*.edf") if "sleepscoring" not in p.stem)
    all_subjects = [s for s in all_subjects if s not in EXCLUDED]

    trainval_subjects, test_subjects = train_test_split(all_subjects, test_size=0.20, random_state=SEED)
    train_subjects, val_subjects = train_test_split(trainval_subjects, test_size=0.20, random_state=SEED)

    split_info = {
        "seed": SEED,
        "train_subjects": sorted(train_subjects),
        "val_subjects": sorted(val_subjects),
        "test_subjects": sorted(test_subjects),
        "n_train": len(train_subjects),
        "n_val": len(val_subjects),
        "n_test": len(test_subjects),
        "all_subjects": all_subjects,
    }
    with open(split_path, "w") as f:
        json.dump(split_info, f, indent=2)
    print("สร้าง split ใหม่")

print("Total:", len(split_info["all_subjects"]))
print("Train:", split_info["n_train"], "| Val:", split_info["n_val"], "| Test:", split_info["n_test"])

In [ ]:
SPLIT_PATH = OUT_DIR / "split_subjects.json"

assert SPLIT_PATH.exists(), f"ไม่พบไฟล์: {SPLIT_PATH}"

with open(SPLIT_PATH, "r") as f:
    split_info = json.load(f)

split_dirs = {
    "train": OUT_DIR / "train",
    "val": OUT_DIR / "val",
    "test": OUT_DIR / "test",
}

expected_subjects = {
    split: set(split_info[f"{split}_subjects"])
    for split in split_dirs
}

file_subjects = {}

for split, folder in split_dirs.items():
    assert folder.exists(), f"ไม่พบโฟลเดอร์: {folder}"

    files = sorted(folder.glob("SN*.npz"))
    file_subjects[split] = {f.stem for f in files}

    unexpected = file_subjects[split] - expected_subjects[split]
    missing = expected_subjects[split] - file_subjects[split]

    print(f"\n{split.upper()}")
    print(f"Subjects ใน split JSON: {len(expected_subjects[split])}")
    print(f"ไฟล์ NPZ ที่พบ: {len(files)}")
    print(f"Subjects ที่ไม่มีไฟล์ NPZ: {sorted(missing)}")
    print(f"Subjects ที่ไม่อยู่ใน split JSON: {sorted(unexpected)}")

    assert not unexpected, (
        f"พบ Subject ที่ไม่ตรงกับ split JSON ใน {split}: {sorted(unexpected)}"
    )

    for file in files:
        with np.load(file, allow_pickle=True) as data:
            assert "subject_id" in data.files, (
                f"ไม่พบ subject_id ในไฟล์ {file.name}"
            )
            stored_subject = str(data["subject_id"].item())

        assert stored_subject == file.stem, (
            f"Subject ID ไม่ตรงกัน: {file.name} มี ID เป็น {stored_subject}"
        )

        assert stored_subject in expected_subjects[split], (
            f"{stored_subject} อยู่ผิดชุด: {split}"
        )

train_val = file_subjects["train"] & file_subjects["val"]
train_test = file_subjects["train"] & file_subjects["test"]
val_test = file_subjects["val"] & file_subjects["test"]

print("\n" + "=" * 45)
print("ตรวจสอบ Subject Leakage")
print("=" * 45)
print(f"Train ∩ Validation: {sorted(train_val)}")
print(f"Train ∩ Test: {sorted(train_test)}")
print(f"Validation ∩ Test: {sorted(val_test)}")

assert not train_val, "พบ Subject ซ้ำระหว่าง Train กับ Validation"
assert not train_test, "พบ Subject ซ้ำระหว่าง Train กับ Test"
assert not val_test, "พบ Subject ซ้ำระหว่าง Validation กับ Test"

print("\nผ่าน: ไม่พบ Subject ซ้ำระหว่างทั้ง 3 ชุด")
print("ผ่าน: Subject ID ในไฟล์ NPZ ตรงกับชื่อไฟล์และ split JSON")
print("หมายเหตุ: Subject ที่ไม่มีไฟล์ NPZ ต้องตรวจสอบสาเหตุเพิ่มเติม")

# Section 3 อ่าน Ground Truth และกำหนดข้อมูลนำเข้า

In [ ]:
def get_labels(subject, n_epochs):
    ann = mne.read_annotations(DATA_DIR / f"{subject}_sleepscoring.edf")
    labels = ["?"] * n_epochs
    onsets = [None] * n_epochs

    for onset, duration, desc in zip(ann.onset, ann.duration, ann.description):
        stage = desc.split()[-1]
        if stage not in STAGES:
            continue
        first = int(round(onset / 30))
        last = int(round((onset + duration) / 30))
        for e in range(first, last):
            if e < n_epochs:
                labels[e] = stage
                onsets[e] = onset + (e - first) * 30
    return labels, onsets


def pick_channels(raw):
    picks = []
    for key in CHANNELS:
        found = [i for i, n in enumerate(raw.ch_names) if key.lower() in n.lower()]
        if len(found) != 1:
            return None
        picks.append(found[0])
    return picks


def find_run(labels, stage):
    count = 0
    for i, lab in enumerate(labels):
        count = count + 1 if lab == stage else 0
        if count == N_CONSEC:
            return i - N_CONSEC + 1, N_CONSEC
    return None, 0


def process_subject(subject, split):
    raw = mne.io.read_raw_edf(DATA_DIR / f"{subject}.edf", preload=False)

    if raw.info["sfreq"] != FS:
        raise ValueError(f"{subject}: sampling rate = {raw.info['sfreq']}")

    picks = pick_channels(raw)
    if picks is None:
        return {
            "subject": subject, "split": split, "n_groups": 0, "n_epochs": 0,
            "complete": False, "missing_stages": "missing channels", "short_stages": ""
        }

    labels, onsets = get_labels(subject, raw.n_times // EPOCH_LEN)
    X, y, stage, epoch_indices = [], [], [], []
    start_samples, end_samples, n_valid = [], [], []
    missing = []

    for s in STAGES:
        first, n = find_run(labels, s)

        if first is None:
            missing.append(s)
            continue

        idx = list(range(first, first + n))
        data_uv = raw.get_data(
            picks=picks,
            start=first * EPOCH_LEN,
            stop=(first + n) * EPOCH_LEN,
            units="uV"
        )
        x = data_uv.T.reshape(n, EPOCH_LEN, len(picks))

        X.append(x)
        y.append([labels[e] for e in idx])
        stage.append(s)
        epoch_indices.append(idx)
        start_samples.append([e * EPOCH_LEN for e in idx])
        end_samples.append([(e + 1) * EPOCH_LEN - 1 for e in idx])
        n_valid.append(n)

    output_file = OUT_DIR / split / f"{subject}.npz"

    if len(X) == len(STAGES) and len(missing) == 0:

        np.savez(
            output_file,
            X=np.array(X, dtype=object),
            y=np.array(y, dtype=object),
            subject_id=subject,
            stage=np.array(stage),
            epoch_indices=np.array(epoch_indices, dtype=object),
            start_samples=np.array(start_samples, dtype=object),
            end_samples=np.array(end_samples, dtype=object),
            n_valid=np.array(n_valid)
        )

        return {
            "subject": subject,
            "split": split,
            "n_groups": 5,
            "n_epochs": 25,
            "complete": True,
            "missing_stages": ""
        }

    if output_file.exists():
        output_file.unlink()

    return {
        "subject": subject,
        "split": split,
        "n_groups": 0,
        "n_epochs": 0,
        "complete": False,
        "missing_stages": ",".join(missing)
    }

# Section 4 คัดเลือกช่วง Epoch ที่ต่อเนื่องกัน

In [ ]:
rows = []

for split in ["train", "val", "test"]:
    for subject in split_info[f"{split}_subjects"]:
        row = process_subject(subject, split)
        rows.append(row)
        print(split, subject, "groups:", row["n_groups"], "missing:", row["missing_stages"])

summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "preprocessing_summary.csv", index=False)

# Section 5 สร้างและตรวจสอบชุดข้อมูล

In [ ]:
ver_rows = []

for _, r in summary[summary["n_groups"] > 0].iterrows():
    subject, split = r["subject"], r["split"]

    data = np.load(OUT_DIR / split / f"{subject}.npz", allow_pickle=True)
    raw = mne.io.read_raw_edf(DATA_DIR / f"{subject}.edf", preload=False)
    picks = pick_channels(raw)
    labels, onsets = get_labels(subject, raw.n_times // EPOCH_LEN)

    for g in range(len(data["stage"])):
        stage = str(data["stage"][g])
        n = int(data["n_valid"][g])
        ep = np.asarray(data["epoch_indices"][g], dtype=int)
        st = np.asarray(data["start_samples"][g], dtype=int)
        en = np.asarray(data["end_samples"][g], dtype=int)
        y = np.asarray(data["y"][g])
        X = np.asarray(data["X"][g])
        where = f"{subject} / {stage} / epochs {list(ep)}"

        assert n == N_CONSEC, f"{where}: n_valid ไม่ครบ {N_CONSEC}: {n}"
        assert len(ep) == n, f"{where}: epoch_indices มี {len(ep)} แต่ n_valid={n}"
        assert len(st) == n, f"{where}: start_samples มี {len(st)} แต่ n_valid={n}"
        assert len(en) == n, f"{where}: end_samples มี {len(en)} แต่ n_valid={n}"
        assert len(y) == n, f"{where}: y มี {len(y)} แต่ n_valid={n}"

        if n > 1:
            assert np.all(np.diff(ep) == 1), f"{where}: epoch ไม่ต่อเนื่อง"

        assert X.shape == (n, EPOCH_LEN, len(picks)), f"{where}: shape ของ X ผิด {X.shape}"
        assert len(set(y)) == 1 and y[0] == stage, f"{where}: y ไม่ใช่ stage เดียวกัน"

        selected_first, selected_n = find_run(labels, stage)
        assert selected_first == int(ep[0]), f"{where}: first epoch ไม่ตรงกับ scoring"
        assert selected_n == n, f"{where}: จำนวน epoch ไม่ตรงกับ scoring"

        for k in range(n):
            e = int(ep[k])
            w = f"{subject} / {stage} / epoch {e}"

            assert st[k] == e * EPOCH_LEN, f"{w}: start sample ผิด"
            assert en[k] == (e + 1) * EPOCH_LEN - 1, f"{w}: end sample ผิด"
            assert en[k] - st[k] + 1 == EPOCH_LEN, f"{w}: จำนวน sample ผิด"
            assert labels[e] == stage, f"{w}: Ground Truth คือ {labels[e]}"
            assert y[k] == labels[e], f"{w}: y ไม่ตรงกับ scoring"
            assert abs(onsets[e] - st[k] / FS) < 1e-6, f"{w}: onset ไม่ตรงกับ sample"


            if k > 0:
                assert st[k] == en[k - 1] + 1, f"{w}: ไม่ต่อจาก epoch ก่อนหน้า"
                assert abs(onsets[e] - onsets[e - 1] - 30) < 1e-6, f"{w}: onset ไม่ห่าง 30 วินาที"

            sig = raw.get_data(
                picks=picks,
                start=int(st[k]),
                stop=int(en[k]) + 1,
                units="uV"
            ).T

            assert X[k].shape == sig.shape, f"{w}: shape ของ Raw Signal ไม่ตรง {X[k].shape} vs {sig.shape}"

            X_k = np.asarray(X[k], dtype=sig.dtype)
            assert np.array_equal(sig, X_k), f"{w}: Raw Signal ไม่ตรงกับ EDF"

            ver_rows.append({
                "Subject": subject,
                "Stage": stage,
                "Epoch": e,
                "Start Sample": int(st[k]),
                "End Sample": int(en[k]),
                "Ground Truth": labels[e],
                "n_consecutive": n
            })

verification = pd.DataFrame(ver_rows)
verification.to_csv(OUT_DIR / "verification.csv", index=False)

print("Verification ผ่านทั้งหมด:", len(verification), "epochs")

# Section 6 ตรวจสอบข้อมูลที่คัดเลือก

In [ ]:
PRINT_SUBJECTS = ["SN001"]

for subject in PRINT_SUBJECTS:
    d = verification[verification["Subject"] == subject]

    for stage in STAGES:
        ds = d[d["Stage"] == stage]

        if len(ds) == 0:
            print(f"Subject: {subject} | Stage: {stage} | ไม่มีข้อมูล\n")
            continue

        n = ds["n_consecutive"].iloc[0]

        print(f"Subject: {subject}")
        print(f"Stage: {stage}")
        print(f"Consecutive Epochs: {n}")
        print("Selected Epoch:", list(ds["Epoch"]))

        for _, r in ds.iterrows():
            print(
                f"Epoch {r['Epoch']} → "
                f"Sample {r['Start Sample']}–{r['End Sample']} → "
                f"Ground Truth {r['Ground Truth']}"
            )

        print()

# Section 7 แสดงตัวอย่างสัญญาณที่คัดเลือก

In [ ]:
PLOT_SUBJECT = "SN001"
PLOT_STAGE = "N2"

split = summary.loc[
    summary["subject"] == PLOT_SUBJECT, "split"
].values[0]

data = np.load(
    OUT_DIR / split / f"{PLOT_SUBJECT}.npz",
    allow_pickle=True
)

g = list(data["stage"]).index(PLOT_STAGE)
n = int(data["n_valid"][g])

X = np.asarray(data["X"][g])
ep = np.asarray(data["epoch_indices"][g], dtype=int)
y = np.asarray(data["y"][g])

X_plot = X.reshape(-1, len(CHANNELS))
t = np.arange(len(X_plot)) / FS

fig, axes = plt.subplots(
    len(CHANNELS), 1,
    figsize=(16, 14),
    sharex=True
)

for i, ax in enumerate(axes):
    ax.plot(t, X_plot[:, i], linewidth=0.5)
    ax.set_ylabel(f"{CHANNEL_NAMES[i]} (µV)")

    for b in range(n + 1):
        ax.axvline(b * 30, color="red", linestyle="--")

for k in range(n):
    axes[0].text(
        k * 30 + 15, 1.02,
        f"Epoch {ep[k]}\n{y[k]}",
        ha="center", va="bottom",
        transform=axes[0].get_xaxis_transform()
    )

axes[-1].set_xlabel("Time (seconds)")
fig.suptitle(
    f"{PLOT_SUBJECT} | {PLOT_STAGE} | {n} consecutive epochs",
    y=1.0
)

plt.show()

# Section 8 สรุปผลการเตรียมข้อมูล

In [ ]:
print("Total Subjects:", len(summary))

short_rows = []

for split in ["train", "val", "test"]:
    d = summary[summary["split"] == split]

    inc = d[d["missing_stages"].fillna("") != ""]

    print(f"\n[{split.upper()}]")
    print("Subjects:", len(d))
    print("มีครบ 5 Stages:", int(d["complete"].sum()))
    print("ไม่ครบ:", len(inc))

    for _, r in inc.iterrows():
        print("  ", r["subject"], "ขาด:", r["missing_stages"])

    short_count = 0

    for _, r in d.iterrows():
        subject = r["subject"]

        if r["n_groups"] == 0:
            continue

        labels, _ = get_labels(
            subject,
            mne.io.read_raw_edf(
                DATA_DIR / f"{subject}.edf",
                preload=False,
                verbose="ERROR"
            ).n_times // EPOCH_LEN
        )

        short_stages = []

        for stage in STAGES:
            if stage not in labels:
                continue

            first, n = find_run(labels, stage)

            if first is None:
                short_stages.append(stage)

        if short_stages:
            short_count += 1
            short_rows.append({
                "split": split,
                "subject": subject,
                "short_stages": ",".join(short_stages)
            })

            print("  ", subject, "สั้น:", ",".join(short_stages))

    print("Subjects ที่มี Stage แต่ไม่มีช่วงต่อเนื่องครบ 5 epochs:", short_count)
    print("Stage groups:", int(d["n_groups"].sum()))
    print("Epochs ที่เก็บจริง:", int(d["n_epochs"].sum()))

print("\n" + "=" * 50)
print("รายละเอียด Stage Groups จากไฟล์ NPZ")
print("=" * 50)

PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")
NPZ_DIR = PROJECT_DIR / "dataset/preprocess_microvolt"

rows = []

for split in ["train", "val", "test"]:
    for file in sorted((NPZ_DIR / split).glob("SN*.npz")):
        with np.load(file, allow_pickle=True) as data:

            for g in range(len(data["stage"])):
                X = np.asarray(data["X"][g])
                y = np.asarray(data["y"][g])
                epochs = np.asarray(data["epoch_indices"][g], dtype=int)

                rows.append({
                    "split": split,
                    "subject": file.stem,
                    "stage": str(data["stage"][g]),
                    "n_valid": int(data["n_valid"][g]),
                    "X_shape": str(X.shape),
                    "y_length": len(y),
                    "epoch_indices": epochs.tolist(),
                    "all_labels_match_stage": bool(
                        len(y) > 0 and np.all(y == data["stage"][g])
                    ),
                })

df = pd.DataFrame(rows)

print("\nจำนวน Stage groups:", len(df))
print("จำนวน Subject:", df["subject"].nunique())

print("\nจำนวนกลุ่มตามจำนวน epochs:")
print(df["n_valid"].value_counts().sort_index())

print("\nจำนวนกลุ่มตาม Split:")
print(df.groupby(["split", "n_valid"]).size().to_string())

print("\nตัวอย่างข้อมูล:")
display(df.head(15))

print("\nกลุ่มที่ไม่ครบ 5 epochs:")
display(df[df["n_valid"] < 5])

print("\nกลุ่มที่มี Label ไม่ตรงกับ Stage:")
display(df[~df["all_labels_match_stage"]])

print("\n" + "=" * 50)
print("สรุปผลการเตรียมข้อมูล")
print("=" * 50)

print("จำนวน Stage groups:", len(df))
print("จำนวน Epochs:", int(df["n_valid"].sum()))
print("จำนวนกลุ่มที่ Label ตรงกับ Stage:",
      int(df["all_labels_match_stage"].sum()))

print("\nSignal unit: µV")
print("Channels:", CHANNEL_NAMES)
print("Maximum consecutive epochs:", N_CONSEC)
print("Samples per epoch:", EPOCH_LEN)
print("Output directory:", NPZ_DIR)

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")
DATA_DIR = PROJECT_DIR / "dataset/preprocess_microvolt"

rows = []

for split in ["train", "val", "test"]:
    for file in sorted((DATA_DIR / split).glob("SN*.npz")):
        data = np.load(file, allow_pickle=True)

        for g in range(len(data["stage"])):
            X = np.asarray(data["X"][g])
            y = np.asarray(data["y"][g])
            epochs = np.asarray(data["epoch_indices"][g], dtype=int)

            rows.append({
                "split": split,
                "subject": file.stem,
                "stage": str(data["stage"][g]),
                "n_valid": int(data["n_valid"][g]),
                "X_shape": str(X.shape),
                "y_length": len(y),
                "epoch_indices": epochs.tolist(),
                "all_labels_match_stage": bool(
                    len(y) > 0 and np.all(y == data["stage"][g])
                ),
            })

df = pd.DataFrame(rows)

print("จำนวน Stage groups:", len(df))
print("จำนวน Subject:", df["subject"].nunique())
print("\nจำนวนกลุ่มตามจำนวน epochs:")
print(df["n_valid"].value_counts().sort_index())

print("\nจำนวนกลุ่มตาม Split:")
print(df.groupby(["split", "n_valid"]).size().to_string())

print("\nตัวอย่างข้อมูล:")
display(df.head(15))

print("\nกลุ่มที่ไม่ครบ 5 epochs:")
display(df[df["n_valid"] < 5])

print("\nกลุ่มที่มี Label ไม่ตรงกับ Stage:")
display(df[~df["all_labels_match_stage"]])